In [3]:
#TODO make sure this renders in the github repo

# Data Loader

The decoder-only model (Llama 3 8B), is trained on next-token prediction using a causal mask: Given a continuous stream of tokens, we slice a fixed-length window (e.g., 256 tokens).

- **Input** ($X$): Tokens $0$ to $N-1$, e.g., $[x_0, x_1, ..., x_{N-1}]$
- **Target** ($Y$): Tokens $1$ to $N$, e.g., $[x_1, x_2, ..., x_{N}]$
- Timestamp example sequence: "This is the Llama 3 8B"
  - The **causal mask** ensures that the tokenized representation for token $i$ is calculated using only the tokens at positions $\leq i$, this prevents the model from "cheating" by looking at the target tokens during training.
  - **Note:** The timestamp table is only for visualization purposes, in practice, the model processes the entire window in one forward pass!

    | Timestamp | Input | Target |
    | --- | --- | --- |
    | 1 | "This" | "is" |
    | 2 | "This is" | "the" |
    | 3 | "This is the" | "Llama" |
    | 4 | "This is the Llama" | "3" |
    | 5 | "This is the Llama 3" | "8B" |

**Example of what the tokens will look like:**

```text
First 10 tokens of the Input:  
    [1, 269, 72, 224, 44, 81, 71, 72, 83, 262]
First 10 tokens of the Target (shifted right): 
    [269, 72, 224, 44, 81, 71, 72, 83, 262, 71]
```

- The dataloader yields dense, packed **batches of tokens**.

In [4]:
# @i-c
%load_ext autoreload
%autoreload 2

In [5]:
import EasyJupyter
import torch
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer

/Users/tonyavis/miniconda3/envs/how_to_create_llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_config import BaseConfig

In [ ]:
class FineWebEduDataset(IterableDataset):
    def __init__(self, cfg: BaseConfig, tokenizer):
        """
        Handles loading the HuggingFace FineWeb-edu dataset.

        Args:
            tokenizer: Either my custom trained BPE tokenizer, or the HuggingFace tokenizer for the Pre-trained Llama 3 models. 
        """
        self.cfg = cfg

        # Load the a subset randomly sampled from the whole dataset of around 10-Billion gpt2 tokens in streaming mode.
        self.dataset = load_dataset(
            "HuggingFaceFW/fineweb-edu",
            name="sample-10BT",
            split="train",
            streaming=True,
        )

        self.tokenizer = tokenizer

        self.doc_end_token = self.tokenizer.token_to_id(
            cfg.special_tokens["doc_end_token"]["token"]
        )

    def __iter__(self):
        # Iterate through the dataset and yield input and target pairs.
        buffer = []

        context_len = self.cfg.context_len # The maximum number of tokens the model can remember.

        for example in self.dataset:
            # Tokenize the raw text document.
            tokens = self.tokenizer.encode(example["text"]).ids

            # Append the document tokens and the doc_end_token separator.
            buffer.extend(tokens)
            buffer.append(self.doc_end_token)

            # Once we have enough tokens for a full context window + 1 (for the target shift)
            while len(buffer) >= context_len + 1:
                # Slice the exact context window
                chunk = buffer[: context_len + 1]

                # Shift the buffer forward so the next context window picks up where this one left off.
                buffer = buffer[context_len :]

                # Input is the first N tokens
                x = torch.tensor(chunk[:-1], dtype=torch.long)
                # Target is shifted 1 token to the right
                y = torch.tensor(chunk[1:], dtype=torch.long)

                yield x, y

In [ ]:
def create_causal_dataloader(cfg: BaseConfig, tokenizer):
    """
    Creates a causal dataloader that yields dense, packed batches of tokens.

    Args:
        tokenizer: Either my custom trained BPE tokenizer, or the HuggingFace tokenizer for the Pre-trained Llama 3 models.
    """

    dataset = FineWebEduDataset(cfg=cfg, tokenizer=tokenizer)

    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
    )

In [9]:
# @i-c
# Test the data loader
from llama_config import Llama3_scaled_down
from model.tokenizer import BPETokenizer

cfg = Llama3_scaled_down()
cfg._setup_folders()

# Get a trained tokenizer
t = BPETokenizer(cfg)
success, tokenizer = t.load_tokenizer()

if not success:
    print("A tokenizer is not saved, please run the test case in the tokenizer.ipynb notebook first, or train a full sized tokenizer!")
    exit()

dataloader = create_causal_dataloader(cfg=cfg, tokenizer=tokenizer)

# Fetch a single batch 
print("\n\nFetching a single batch from FineWeb-Edu...")
data_iterator = iter(dataloader)
x,  y = next(data_iterator)
print("\n\nA single batch of data:")
print(f"x: {x}")
print(f"\ny: {y}")

print("\n\nDecoding the first 50 tokens of x[0] to view the document:")
token_ids_list = x[0][:50].tolist()
decoded_document = tokenizer.decode(token_ids_list)
print(f"decoded_document: [{decoded_document}...]")


# Verify the shapes match the llama cfg
print(f"\n\nInput 'x' shape: {x.shape} -> Expected shape: (Batch size, Context length)")
print(f"Input 'y' shape: {y.shape} -> Expected shape: (Batch size, Context length)")

# Verify the target shape are shifted correctly
print(f"\n\nFirst 10 tokens of x[0]: {x[0][:10].tolist()}")
print(f"\nFirst 10 tokens of y[0]: {y[0][:10].tolist()}")


Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM
/Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/utils

Loading existing trained BPE Tokenizer from: (/Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/saved_models/tokenizer/BPE_tokenizer_trained.json)...




Fetching a single batch from FineWeb-Edu...


A single batch of data:
x: tensor([[  1, 269,  72,  ...,  71,  68,  92],
        [ 17, 224,  43,  ...,  69,  82,  82],
        [ 78, 224,  36,  ...,  81,  72, 224],
        ...,
        [ 87, 261,  82,  ...,  87,  88, 271],
        [ 87, 224,  37,  ...,  68,  87,  76],
        [ 82,  81,  17,  ...,  88,  70, 263]])

y: tensor([[269,  72, 224,  ...,  68,  92,  17],
        [224,  43, 275,  ...,  82,  82,  78],
        [224,  36,  81,  ...,  72, 224,  41],
        ...,
        [261,  82, 224,  ...,  88, 271,  87],
        [224,  37,  85,  ...,  87,  76,  82],
        [ 81,  17, 224,  ...,  70, 263, 306]])


Decoding the first 50 tokens of x[0] to view the document:
decoded_document: [The Independent Jane
For all the love, romance and scanda...]


Input 'x' shape: torch.Size([16, 256]) -> Expected shape: (Batch size, Context length)
Input 'y' shape: torch.Size([16, 256]) -> Expected shape: (Batch size, Context length)


First 10 tokens of x[